<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/GPT_TO_GGUF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Sat Oct 31 01:46:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import gc
import hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from transformers import AutoModelForCausalLM, AutoTokenizer

# =====================================================================
# 1. SEED & REPRODUCIBILITY CONFIGURATION
# =====================================================================
SEED = 123
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

GOVERNED_REPO_ID = "frankmorales2020/gpt-oss-20b-topo-governed-cbp-new-version2"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Purge any stale objects from VRAM/RAM
for _name in ("trainer", "topo_opt", "model", "base_model", "embedding_layer"):
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# =====================================================================
# 2. CHECKPOINT RETRIEVAL & BASE SUBSTRATE INITIALIZATION
# =====================================================================
print(f"Downloading checkpoint from {GOVERNED_REPO_ID}...")
checkpoint_file = hf_hub_download(repo_id=GOVERNED_REPO_ID, filename="certified_topological_best.pt")
ckpt = torch.load(checkpoint_file, map_location="cpu")

BASE_MODEL_ID = ckpt["base_model_id"]
TASK_ORDER = ckpt["task_order"]
TASKS = ckpt["tasks"]
MAX_LENGTH = ckpt["max_length"]
PRIME_ANCHORS = ckpt["prime_anchors"]

print(f"Loading base substrate: {BASE_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
base_model.eval()
embedding_layer = base_model.get_input_embeddings()
input_device = embedding_layer.weight.device
hidden_size = embedding_layer.weight.shape[1]

# =====================================================================
# 3. BASELINE INTEGRITY AUDIT (PRE-INFERENCE)
# =====================================================================
def compute_anchor_digest(embedding_layer, anchors):
    w = embedding_layer.weight.data
    rows = [w[i] for i in anchors if i < w.shape[0]]
    blob = torch.cat([r.flatten().float() for r in rows]).detach().cpu().numpy().tobytes()
    return hashlib.sha256(blob).hexdigest()

recomputed_initial_digest = compute_anchor_digest(embedding_layer, PRIME_ANCHORS)
print(f"Digest stored in checkpoint  : {ckpt['state_hash'][:16]}")
print(f"Digest recomputed from model : {recomputed_initial_digest[:16]}")
assert recomputed_initial_digest == ckpt["state_hash"], (
    "Integrity failure: Base model substrate does not match checkpoint digest."
)
assert hidden_size == ckpt["hidden_size"], "Hidden size mismatch between substrate and checkpoint."
print("Integrity audit              : PASS - Anchor coordinate rows verified.\n")

# =====================================================================
# 4. HEAD RECONSTRUCTION & STRICT WEIGHT BINDING
# =====================================================================
def make_head(hidden_dim):
    return nn.Sequential(nn.LayerNorm(hidden_dim), nn.Linear(hidden_dim, 2)).float()

heads = nn.ModuleDict({f"classifier_{t}": make_head(hidden_size) for t in TASK_ORDER})
heads.load_state_dict(ckpt["task_heads_state_dict"], strict=True)
heads.to(DEVICE).eval()


def pool_last_token(hidden_states, attention_mask):
    positions = torch.arange(attention_mask.shape[1], device=attention_mask.device)
    last_idx = (attention_mask.long() * positions).argmax(dim=1)
    batch_idx = torch.arange(hidden_states.shape[0], device=hidden_states.device)
    return hidden_states[batch_idx, last_idx.to(hidden_states.device), :]


@torch.no_grad()
def predict(texts, task):
    enc = tokenizer(
        texts,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    enc = {k: v.to(input_device) for k, v in enc.items()}
    out = base_model(
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
        output_hidden_states=True,
        return_dict=True
    )
    pooled = pool_last_token(out.hidden_states[-1], enc["attention_mask"])
    with torch.autocast(device_type=DEVICE.type, enabled=False):
        logits = heads[f"classifier_{task}"](pooled.to(device=DEVICE, dtype=torch.float32))
    return F.softmax(logits, dim=-1).cpu()

# =====================================================================
# 5. HELD-OUT TEST EVALUATION
# =====================================================================
raw_test = load_dataset("SetFit/ag_news", split=ckpt["test_split"])
print(f"--- Test accuracy on SetFit/ag_news {ckpt['test_split']} ---")

live_accuracy = {}
for task in TASK_ORDER:
    mapping = {orig: new for new, orig in enumerate(TASKS[task]["labels"])}
    pairs = [(t, mapping[l]) for t, l in zip(raw_test["text"], raw_test["label"]) if l in mapping]
    correct = 0
    for i in range(0, len(pairs), 32):
        chunk = pairs[i:i + 32]
        probs = predict([t for t, _ in chunk], task)
        correct += (probs.argmax(dim=-1) == torch.tensor([y for _, y in chunk])).sum().item()
    acc = correct / len(pairs)
    live_accuracy[task] = acc
    print(f"Task {task} ({TASKS[task]['names'][0]} vs {TASKS[task]['names'][1]}): "
          f"{acc:.2%} on {len(pairs)} samples | recorded at training time: {ckpt['final_accuracy'][task]:.2%} | "
          f"majority class: {ckpt['majority_accuracy'][task]:.2%}")

# =====================================================================
# 6. IN-DOMAIN TEST PROMPT VERIFICATION
# =====================================================================
queries = {
    'A': [("International diplomatic summits opened in Geneva to discuss regional peace treaties.", "World"),
          ("The national football championship ended in an overtime thriller last night.", "Sports")],
    'B': [("Wall Street indexes rallied to record highs following positive corporate earnings.", "Business"),
          ("A revolutionary quantum computing chip demonstrates exponential speedup in cryptographic calculations.", "Sci/Tech")],
    'C': [("Peace negotiators from both governments met in the capital to discuss a ceasefire.", "World"),
          ("Space exploration agencies successfully placed a new observation satellite into orbit.", "Sci/Tech")],
}

print()
for task in TASK_ORDER:
    names = TASKS[task]["names"]
    print(f"--- Task head {task}: {names[0]} (class 0) vs {names[1]} (class 1) ---")
    for text, expected in queries[task]:
        probs = predict([text], task)[0]
        pred = int(probs.argmax())
        verdict = "correct" if names[pred] == expected else "WRONG"
        print(f"Query      : \"{text}\"")
        print(f"Prediction : {names[pred]} (class {pred}) | confidence {probs[pred].item():.2%} | "
              f"expected {expected} -> {verdict}\n")

# =====================================================================
# 7. TELEMETRY AUDIT: DRIFT & FORGETTING (FGT) HARD SILICON ASSERTIONS
# =====================================================================
print("=" * 80)
print("TELEMETRY AUDIT: RIGOROUS INVARIANCE & ZERO-FORGETTING ASSERTIONS")
print("=" * 80)

# --- 1. Compute & Assert Parameter Drift (Δdrift) on Prime Coordinates ---
w_tensor = embedding_layer.weight.data
computed_final_digest = compute_anchor_digest(embedding_layer, PRIME_ANCHORS)
stored_digest = ckpt["state_hash"]
digest_verified = (computed_final_digest == stored_digest)

if "anchor_rows_data" in ckpt:
    stored_tensors = ckpt["anchor_rows_data"].to(w_tensor.device)
    max_drift = torch.max(torch.abs(w_tensor[PRIME_ANCHORS] - stored_tensors)).item()
else:
    # Direct bitwise invariance check from live memory
    max_drift = 0.0 if digest_verified else float("inf")

# --- 2. Compute & Assert Catastrophic Forgetting (FGT) ---
# Forgetting per task = max(0.0, recorded_training_acc - live_eval_acc)
fgt_per_task = {}
for task in TASK_ORDER:
    recorded_acc = ckpt["final_accuracy"][task]
    current_acc = live_accuracy[task]
    fgt_per_task[task] = max(0.0, recorded_acc - current_acc)

computed_mean_fgt = float(np.mean(list(fgt_per_task.values())))

print(f"Prime Coordinates Audited   : {PRIME_ANCHORS}")
print(f"Stored Checkpoint Hash      : {stored_digest}")
print(f"Recomputed Post-Run Hash    : {computed_final_digest}")
print(f"Max Parameter Drift (Δdrift): {max_drift:.12f}")
for task in TASK_ORDER:
    print(f"• Task {task} Forgetting (FGT)      : {fgt_per_task[task]:.4f} "
          f"(Trained: {ckpt['final_accuracy'][task]:.2%}, Live: {live_accuracy[task]:.2%})")
print(f"Mean Catastrophic Forgetting: {computed_mean_fgt:.6f}")
print("-" * 80)

# Hard invariant gates: Any positive drift or accuracy degradation halts execution
assert digest_verified, (
    f"SILICON DRIFT BREACH: Recomputed hash ({computed_final_digest}) "
    f"does not match baseline ({stored_digest})."
)
assert max_drift == 0.0, f"SILICON DRIFT BREACH: Δdrift = {max_drift:.12f} > 0.0"

assert computed_mean_fgt == 0.0, (
    f"CATASTROPHIC FORGETTING BREACH: Mean FGT = {computed_mean_fgt:.6f} > 0.0. "
    f"Task metrics: {fgt_per_task}"
)

print("AUDIT RESULT: PASSED — Zero Silicon Drift & Zero Catastrophic Forgetting Certified.")
print("=" * 80)

Loading base substrate: openai/gpt-oss-20b...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] MXFP4 quantization requires the `kernels` package: Please install a compatible version (0.17.2 <= version < 0.18.0), e.g. `pip install kernels==0.17.2`We will default to dequantizing the model to bf16.


Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

Digest stored in checkpoint  : 334ea0c8ca2e9af5
Digest recomputed from model : 334ea0c8ca2e9af5
Integrity audit              : PASS - Anchor coordinate rows verified.

--- Test accuracy on SetFit/ag_news test[:2000] ---
Task A (World vs Sports): 96.53% on 1037 samples | recorded at training time: 96.34% | majority class: 50.72%
Task B (Business vs Sci/Tech): 88.99% on 963 samples | recorded at training time: 88.58% | majority class: 53.37%
Task C (World vs Sci/Tech): 95.51% on 1025 samples | recorded at training time: 95.41% | majority class: 50.15%

--- Task head A: World (class 0) vs Sports (class 1) ---
Query      : "International diplomatic summits opened in Geneva to discuss regional peace treaties."
Prediction : World (class 0) | confidence 100.00% | expected World -> correct

Query      : "The national football championship ended in an overtime thriller last night."
Prediction : Sports (class 1) | confidence 99.45% | expected Sports -> correct

--- Task head B: Business (class 0

## gpt2gguf

In [ ]:
import os, sys, subprocess
from pathlib import Path

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

# ---- one-time setup (runs once, then restarts the runtime) ----
if not os.path.exists("deps_v2.flag"):
    sh("pip install -q -U sentencepiece protobuf numpy")
    sh("test -d llama.cpp || git clone --depth 1 https://github.com/ggml-org/llama.cpp")
    sh("pip install -q --no-deps ./llama.cpp/gguf-py")   # NOT llama.cpp/requirements.txt
    sh("cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF > /dev/null")
    sh("cmake --build llama.cpp/build --target llama-quantize -j > /dev/null")
    open("deps_v2.flag", "w").write("ok")
    print("Setup done. Runtime is restarting: run this cell again.")
    os.kill(os.getpid(), 9)

In [1]:
import subprocess
subprocess.run("cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF", shell=True, check=True)
subprocess.run("cmake --build llama.cpp/build --config Release -j", shell=True, check=True)

CompletedProcess(args='cmake --build llama.cpp/build --config Release -j', returncode=0)

In [2]:
!rm -rf gpt-oss-20b-topo-governed-*

In [3]:
# =====================================================================
# 2. RESOLVE BASE SUBSTRATE AND CONVERT DIRECTLY TO GGUF
# =====================================================================
import os
import subprocess
import torch
from huggingface_hub import hf_hub_download, snapshot_download

GOVERNED_REPO_ID = "frankmorales2020/gpt-oss-20b-topo-governed-cbp-new-version2"

print(f"[*] Downloading checkpoint from {GOVERNED_REPO_ID}...")
ckpt_file = hf_hub_download(repo_id=GOVERNED_REPO_ID, filename="certified_topological_best.pt")
ckpt = torch.load(ckpt_file, map_location="cpu")

BASE_MODEL_ID = ckpt["base_model_id"]
print(f"[*] Resolved base substrate: {BASE_MODEL_ID}")

# Download full base weights and tokenizer configuration
print(f"[*] Downloading full base model snapshot for {BASE_MODEL_ID}...")
model_dir = snapshot_download(repo_id=BASE_MODEL_ID)

# =====================================================================
# 3. RUN CONVERT_HF_TO_GGUF.PY WITH --no-lazy (FORCES UNPACK TO BF16)
# =====================================================================
bf16_gguf = "/content/gpt-oss-20b-topo-governed-bf16.gguf"
q4km_gguf = "/content/gpt-oss-20b-topo-governed-Q4_K_M.gguf"
convert_script = "/content/llama.cpp/convert_hf_to_gguf.py"

print(f"[*] Converting {model_dir} to {bf16_gguf} via {convert_script}...")
conv_cmd = [
    "python", convert_script,
    model_dir,
    "--outfile", bf16_gguf,
    "--outtype", "bf16",
    "--no-lazy"
]

conv_res = subprocess.run(conv_cmd, capture_output=True, text=True)

if conv_res.returncode != 0:
    print(conv_res.stdout)
    print(conv_res.stderr)
    raise RuntimeError(f"convert_hf_to_gguf.py failed with returncode {conv_res.returncode}")

print(f"[+] Successfully generated BF16 GGUF: {bf16_gguf}")

[*] Downloading checkpoint from frankmorales2020/gpt-oss-20b-topo-governed-cbp-new-version2...


certified_topological_best.pt: reconstructing file:   0%|          |  0.00B /  145kB            

certified_topological_best.pt: downloading bytes:           |  0.00B            

[*] Resolved base substrate: openai/gpt-oss-20b
[*] Downloading full base model snapshot for openai/gpt-oss-20b...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

[*] Converting /root/.cache/huggingface/hub/models--openai--gpt-oss-20b/snapshots/6cee5e81ee83917806bbde320786a8fb61efebee to /content/gpt-oss-20b-topo-governed-bf16.gguf via /content/llama.cpp/convert_hf_to_gguf.py...
[+] Successfully generated BF16 GGUF: /content/gpt-oss-20b-topo-governed-bf16.gguf


In [4]:
# =====================================================================
# 4. QUANTIZE TO STANDALONE GGUF (ALLOWING MXFP4 EXPERTS)
# =====================================================================
quant_bin = "/content/llama.cpp/build/bin/llama-quantize"

assert os.path.exists(quant_bin) and os.access(quant_bin, os.X_OK), (
    f"Executable not found or not runnable at: {quant_bin}"
)

print(f"[*] Quantizing {bf16_gguf} -> {q4km_gguf} via {quant_bin}...")
quant_cmd = [
    quant_bin,
    "--allow-requantize",
    bf16_gguf,
    q4km_gguf,
    "Q4_K_M"
]

quant_res = subprocess.run(quant_cmd, capture_output=True, text=True)

if quant_res.returncode != 0:
    print("\n" + "=" * 60)
    print("QUANTIZE FAILURE (STDOUT):")
    print(quant_res.stdout)
    print("-" * 60)
    print("QUANTIZE FAILURE (STDERR):")
    print(quant_res.stderr)
    print("=" * 60)
    raise RuntimeError(f"Quantization failed with exit status {quant_res.returncode}")

print(f"\n[+] Quantization succeeded! Output file: {q4km_gguf}")
subprocess.run(["ls", "-lh", q4km_gguf])

[*] Quantizing /content/gpt-oss-20b-topo-governed-bf16.gguf -> /content/gpt-oss-20b-topo-governed-Q4_K_M.gguf via /content/llama.cpp/build/bin/llama-quantize...

[+] Quantization succeeded! Output file: /content/gpt-oss-20b-topo-governed-Q4_K_M.gguf


CompletedProcess(args=['ls', '-lh', '/content/gpt-oss-20b-topo-governed-Q4_K_M.gguf'], returncode=0)

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!mkdir -p /content/drive/MyDrive/gpt-oss-20b-topo

In [7]:
!cp -pr *.gguf /content/drive/MyDrive/gpt-oss-20b-topo/

In [8]:
!ls -ltha /content/drive/MyDrive/gpt-oss-20b-topo/

total 28G
drwx------  2 root root 4.0K Oct 31 02:35 .
drwx------ 48 root root 4.0K Oct 31 02:33 ..
-rw-------  1 root root  15G Oct 31 02:31 gpt-oss-20b-topo-governed-Q4_K_M.gguf
-rw-------  1 root root  13G Oct 31 02:30 gpt-oss-20b-topo-governed-bf16.gguf


## inference-local

In [2]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu \
  --no-cache-dir

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.1/27.1 MB 78.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [llama-cpp-python]


In [13]:
from llama_cpp import Llama

gguf_path = "/content/gpt-oss-20b-topo-governed-Q4_K_M.gguf"

print(f"[*] Initializing Llama runtime for {gguf_path}...")

# Load model with conservative context and threads to prevent memory spikes
llm = Llama(
    model_path=gguf_path,
    n_ctx=512,        # Restrict context window to preserve RAM
    n_threads=4,      # Set according to available CPU cores
    n_gpu_layers=0,   # Pure CPU silicon execution
    use_mmap=False,   # Avoid aggressive virtual memory page faults
    verbose=False
)

test_prompt = "A revolutionary quantum computing chip demonstrates exponential speedup in cryptographic calculations."

print(f"\n[*] Running inference prompt:\n{test_prompt}\n")
output = llm(
    test_prompt,
    max_tokens=64,
    temperature=0.2,
    top_p=0.95,
    stop=["\n\n", "User:"]
)

generated_text = output["choices"][0]["text"]
print("=" * 60)
print("GENERATION OUTPUT:")
print("=" * 60)
print(test_prompt + generated_text)
print("=" * 60)

[*] Initializing Llama runtime for /content/gpt-oss-20b-topo-governed-Q4_K_M.gguf...


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)



[*] Running inference prompt:
A revolutionary quantum computing chip demonstrates exponential speedup in cryptographic calculations.

GENERATION OUTPUT:
A revolutionary quantum computing chip demonstrates exponential speedup in cryptographic calculations. The chip, developed by a team of researchers at the University of California, uses a novel approach to quantum computing that allows it to perform calculations that would take classical computers thousands of years to complete in just a few seconds. The chip is based on a new type of quantum bit, or qubit, that can exist in


## hf

In [14]:
from google.colab import userdata
from huggingface_hub import HfApi

# Retrieve your token from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')
user_or_name = 'frankmorales2020'

REPO_ID = f"{user_or_name}/gpt-oss-20b-topo-governed-cbp-new-version2-gguf"
LOCAL_FILE = "/content/gpt-oss-20b-topo-governed-Q4_K_M.gguf"

api = HfApi(token=HF_TOKEN)

# Ensure the dedicated GGUF repository exists
print(f"[*] Ensuring repository {REPO_ID} exists...")
api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    exist_ok=True,
    private=False
)

# Upload the standalone Q4_K_M binary
print(f"[*] Initializing upload of {LOCAL_FILE} to {REPO_ID}...")
api.upload_file(
    path_or_fileobj=LOCAL_FILE,
    path_in_repo="gpt-oss-20b-topo-governed-Q4_K_M.gguf",
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Release verified standalone Q4_K_M GGUF binary for CPU execution"
)

print(f"[+] Artifact successfully published to: https://huggingface.co/{REPO_ID}")

[*] Ensuring repository frankmorales2020/gpt-oss-20b-topo-governed-cbp-new-version2-gguf exists...
[*] Initializing upload of /content/gpt-oss-20b-topo-governed-Q4_K_M.gguf to frankmorales2020/gpt-oss-20b-topo-governed-cbp-new-version2-gguf...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...topo-governed-Q4_K_M.gguf:   0%|          | 20.8MB / 15.8GB            

Validating                    :           |   0%            

[+] Artifact successfully published to: https://huggingface.co/frankmorales2020/gpt-oss-20b-topo-governed-cbp-new-version2-gguf


## inference-hf

In [3]:
import os
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# 1. Configuration
REPO_ID = "frankmorales2020/gpt-oss-20b-topo-governed-cbp-new-version2-gguf"
FILENAME = "gpt-oss-20b-topo-governed-Q4_K_M.gguf"

print(f"[*] Fetching {FILENAME} from {REPO_ID}...")
model_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    local_dir="/content/models"
)
print(f"[+] Download complete: {model_path}")

# 2. Initialize Runtime
print("[*] Loading GGUF model into memory...")
llm = Llama(
    model_path=model_path,
    n_ctx=1024,       # Adjust context length as needed
    n_threads=4,      # Set to match host CPU physical cores
    n_gpu_layers=0,   # Set > 0 if running with CUDA / GPU offload
    use_mmap=False,   # Avoid memory mapping spikes in virtualized containers
    verbose=False
)

# 3. Deterministic Inference Execution
test_prompt = "A revolutionary quantum computing chip demonstrates exponential speedup in cryptographic calculations."

print(f"\n[*] Prompt:\n{test_prompt}\n")
response = llm(
    test_prompt,
    max_tokens=96,
    temperature=0.2,
    top_p=0.95,
    stop=["\n\n", "User:"]
)

generated_text = response["choices"][0]["text"]

print("=" * 60)
print("INFERENCE RESULT:")
print("=" * 60)
print(test_prompt + generated_text)
print("=" * 60)

[*] Fetching gpt-oss-20b-topo-governed-Q4_K_M.gguf from frankmorales2020/gpt-oss-20b-topo-governed-cbp-new-version2-gguf...


gpt-oss-20b-topo-governed-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 15.8GB            

gpt-oss-20b-topo-governed-Q4_K_M.gguf: downloading bytes:           |  0.00B            

[+] Download complete: /content/models/gpt-oss-20b-topo-governed-Q4_K_M.gguf
[*] Loading GGUF model into memory...


llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)



[*] Prompt:
A revolutionary quantum computing chip demonstrates exponential speedup in cryptographic calculations.

INFERENCE RESULT:
A revolutionary quantum computing chip demonstrates exponential speedup in cryptographic calculations. The chip, developed by a team of researchers at the University of California, uses a novel approach to quantum computing that allows it to perform calculations that would take classical computers thousands of years to complete in just a few seconds. The chip is based on a new type of quantum bit, or qubit, that can exist in multiple states at once, allowing it to perform many calculations simultaneously. The chip is also designed to be highly stable and resistant to errors, making it a promising candidate
